# MedVision-CXR4 — 4-class chest X-ray study (run notebook)

**Task:** COVID-19 · Lung Opacity · Normal · Viral Pneumonia
**Data:** COVID-19 Radiography Database (Kaggle `tawsifurrahman/covid19-radiography-database`, 21,165 images + lung masks)
**Models:** ResNet-50, DenseNet-121, EfficientNet-B0, ConvNeXt-Tiny, ViT-B/16, Swin-T (ImageNet-pretrained) + soft-voting ensembles

This notebook runs everything through one script, `src/run_study.py`. Each stage is one session:

| STAGE | What it runs | Time on Kaggle T4 |
|---|---|---|
| `smoke` | every step on a few batches, to check that everything works | ~20 min |
| `main` | E1 benchmark (6 models × seeds), E2 ensembles, E4 Grad-CAM lung audit | ~9–11 h |
| `audits` | E3 leakage, E5 shortcut (masked lungs), E7 ablations | ~9–11 h |
| `analysis` | merges the runs and builds tables, significance tests and figures | ~5 min |

**On Kaggle:** set Accelerator to *GPU T4 x2* and turn Internet *On*. Set `STAGE` in the next cell, then *Save Version → Save & Run All*.
For `analysis`, first attach the `main` and `audits` versions as inputs (*Add Input → Your Work*).
**`DEMO = True`** runs the whole pipeline on a tiny synthetic dataset on CPU in a few minutes. It is only a check that the code works; its numbers mean nothing.

*Research prototype — not a medical device.*

In [ ]:
# ---- settings ----
STAGE  = "smoke"     # "smoke" | "main" | "audits" | "analysis"
SEEDS  = [0, 1, 2]
EPOCHS = 10
HOURS  = 11.0        # stop starting new runs after this (a Kaggle session is limited to 12 h)
DEMO   = False       # True = synthetic data on CPU, offline (code check only)
CODE_REPO = "https://github.com/Komal467-ai/MedVision-AI.git"   # team repo; code is in its m3_cxr4/ folder (used only if the code isn't already here)

In [ ]:
# ---- get the code ----
import os, sys, glob, subprocess
from pathlib import Path

def find_repo():
    for p in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/medvision-cxr4")]:
        if (p / "src" / "run_study.py").exists():
            return p
    hit = glob.glob("/kaggle/input/**/src/run_study.py", recursive=True)
    return Path(hit[0]).parents[1] if hit else None

REPO = find_repo()
if REPO is None:
    REPO = Path("/kaggle/working/medvision-cxr4") if Path("/kaggle").exists() else Path("medvision-cxr4").resolve()
    subprocess.run(["git", "clone", "-q", CODE_REPO, str(REPO)], check=True)
    hit = sorted(REPO.glob("**/src/run_study.py"))          # code may sit in a subfolder (m3_cxr4/)
    REPO = hit[0].parents[1]
os.chdir(REPO); sys.path.insert(0, str(REPO))
print("code at", REPO)

In [ ]:
# ---- install and run the tests (20 offline tests on synthetic data) ----
!pip install -q -r requirements.txt 2>&1 | grep -v -i -E "warning|already" || true
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU (fine for DEMO)"
!python -m pytest -q 2>&1 | tail -3

In [ ]:
# ---- data ----
EXTRA = []
if DEMO:
    # same folder layout as the Kaggle dataset, generated with the test fixtures
    import cv2, numpy as np
    from tests.conftest import fake_cxr, CLASS_DIRS
    DATA = Path("demo_data/COVID-19_Radiography_Dataset"); rng = np.random.default_rng(0)
    for k, c in enumerate(CLASS_DIRS):
        (DATA / c / "images").mkdir(parents=True, exist_ok=True); (DATA / c / "masks").mkdir(parents=True, exist_ok=True)
        for i in range(30):
            img, mask = fake_cxr(rng, k)
            cv2.imwrite(str(DATA / c / "images" / f"{c}-{i+1}.png"), img)
            cv2.imwrite(str(DATA / c / "masks" / f"{c}-{i+1}.png"), mask)
    DATA_ROOT = str(DATA.parent)
    EXTRA = ["--offline", "--workers", "0", "--bs", "8", "--input_glob", "no_inputs/*"]
    if STAGE != "analysis": STAGE = "smoke"
else:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")
print("data:", DATA_ROOT)

from src.cxrdata.dataset import prepare_table, duplicate_report, leakage_count
t = prepare_table(DATA_ROOT, "group", cache_dir="runs/cache")
print(duplicate_report(t))
print(t.groupby(["split", "label"]).size().unstack())
print("near-duplicate groups crossing splits:", leakage_count(t))

In [ ]:
# ---- run the selected stage (finished runs are skipped, so a stage can be resumed) ----
seeds = " ".join(map(str, SEEDS))
!python -W ignore -m src.run_study --stage {STAGE} --seeds {seeds} --epochs {EPOCHS} --hours {HOURS} --data_root "{DATA_ROOT}" {" ".join(EXTRA)} 2>&1 | grep -v -i warn

## Results
Tables and figures appear once the `analysis` stage (or `smoke`/`DEMO`) has run.

In [ ]:
from IPython.display import Markdown, Image, display
res = Path("results")
for f in ["table_main.md"]:
    if (res / f).exists(): display(Markdown((res / f).read_text()))
for f in sorted(res.glob("run_summary_*.json")): print(f.name, f.read_text())
for f in sorted(res.glob("*.png")):
    print(f.name); display(Image(filename=str(f), width=700))

In [ ]:
# ---- package outputs (on Kaggle: Output tab -> download). Checkpoints (.pth) are left out because they are large ----
!zip -qr ../cxr4_results.zip results runs -x "*.pth" -x "runs/cache/*" && ls -lh ../cxr4_results.zip